# Deploy r-1 from AWS Marketplace

This notebook creates a model from the Reducto r-1 model package, runs real-time inference and a batch transform, and deletes the resources.

r-1 takes one page image and returns the page text as layout blocks. See [../README.md](../README.md) for the request and response contract.

Prerequisites: an IAM role with SageMaker access, and quota for `ml.g6e.2xlarge` (endpoint and transform job).

Contents:
1. Subscribe to the model package
2. Create a model from the model package
3. Real-time inference
4. Batch transform
5. Clean up

## 1. Subscribe to the model package

1. Open the [r-1 listing](<listing URL>) in AWS Marketplace.
2. Select **Continue to Subscribe**, then accept the offer.
3. Select **Continue to configuration** and choose your region. The console shows the model package ARN.

## 2. Create a model from the model package

In [ ]:
%pip install -q "sagemaker>=2,<3"

In [ ]:
import base64
import json
from pathlib import Path

import boto3
import sagemaker
from sagemaker import ModelPackage, get_execution_role
from sagemaker.predictor import Predictor
from sagemaker.utils import name_from_base

# TODO: replace with the model package ARN for each region in the listing.
model_package_map = {
    "us-east-2": "arn:aws:sagemaker:us-east-2:<account>:model-package/<name>",
}

session = sagemaker.Session()
region = session.boto_region_name
if region not in model_package_map:
    raise ValueError(f"r-1 is not available in {region}. Use one of: {sorted(model_package_map)}")

role = get_execution_role()
instance_type = "ml.g6e.2xlarge"
samples = Path("../samples")

model = ModelPackage(
    role=role,
    model_package_arn=model_package_map[region],
    sagemaker_session=session,
)

## 3. Real-time inference

The endpoint takes a few minutes to start.

In [ ]:
endpoint_name = name_from_base("r-1")
model.deploy(initial_instance_count=1, instance_type=instance_type, endpoint_name=endpoint_name)

In [ ]:
runtime = boto3.client("sagemaker-runtime", region_name=region)

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=(samples / "invoice.request.json").read_bytes(),
)
result = json.loads(response["Body"].read())
print(result["finish_reason"], result["usage"])
print(result["text"])

To send your own page, base64-encode a PNG or JPEG:

In [ ]:
def request_body(image_path: Path) -> str:
    return json.dumps({"image_b64": base64.b64encode(image_path.read_bytes()).decode()})


response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=request_body(samples / "rotated.png"),
)
print(json.loads(response["Body"].read())["text"])

## 4. Batch transform

Each S3 object holds one JSON request body. The transform writes one `<input>.out` object for each input.

In [ ]:
bucket = session.default_bucket()
prefix = name_from_base("r-1-batch")
s3 = boto3.client("s3", region_name=region)

for name in ["invoice", "text", "rotated"]:
    s3.put_object(
        Bucket=bucket,
        Key=f"{prefix}/input/{name}.json",
        Body=request_body(samples / f"{name}.png"),
        ContentType="application/json",
    )

transformer = model.transformer(
    instance_count=1,
    instance_type=instance_type,
    strategy="SingleRecord",
    max_concurrent_transforms=1,
    max_payload=6,
    accept="application/json",
    output_path=f"s3://{bucket}/{prefix}/output",
)
transformer.transform(
    data=f"s3://{bucket}/{prefix}/input",
    content_type="application/json",
    split_type=None,
    wait=True,
    logs=False,
)

In [ ]:
for obj in s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/output/")["Contents"]:
    output = json.loads(s3.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read())
    print(f"== {obj['Key']} ({output['finish_reason']})")
    print(output["text"])

## 5. Clean up

Delete the endpoint and the model to stop charges.

In [ ]:
Predictor(endpoint_name, sagemaker_session=session).delete_endpoint(delete_endpoint_config=True)
model.delete_model()

To unsubscribe, open **AWS Marketplace > Manage subscriptions** in the console and cancel the r-1 subscription. Delete all endpoints and models from the model package first.